# ✍️ Agent 1 — Prompt Engineer
## Generates Playwright scraper scripts per competitor

**What this agent does:**
- Reads `input_schema.json`
- Generates a Playwright Python script per competitor
- Validates syntax before saving
- Outputs `scripts/` folder + `scrape_manifest.json`

**SDK: Anthropic (Claude) — NOT OpenAI**

**Next → Agent 2 (Scraper Runner)**

## 1. Install Dependencies

In [ ]:
%pip install anthropic python-dotenv playwright --quiet
# First time only: playwright install chromium

## 2. Setup

In [ ]:
import os, json, ast
from pathlib import Path
from datetime import datetime
from dotenv import load_dotenv
import anthropic

load_dotenv()
print('ANTHROPIC_API_KEY loaded:', bool(os.getenv('ANTHROPIC_API_KEY')))

BASE_DIR    = Path('.')
SCRIPTS_DIR = BASE_DIR / 'scripts'
DATA_DIR    = BASE_DIR / 'data' / 'raw'
REPORTS_DIR = BASE_DIR / 'reports'

for d in [SCRIPTS_DIR, DATA_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

client = anthropic.Anthropic()   # reads ANTHROPIC_API_KEY from .env
print('Anthropic client ready')

## 3. Load Input Schema

In [ ]:
with open('input_schema.json') as f:
    INPUT = json.load(f)

print('Business    :', INPUT['business']['name'])
print('Competitors :', [c['name'] for c in INPUT['competitors']])
if INPUT.get('scrape_targets'):
    print('Scraping    :', [k for k, v in INPUT['scrape_targets'].items() if v])

## 4. Define Tools (plain Python functions + Anthropic tool dicts)

In [ ]:
# ── Tool functions ──────────────────────────────────────────────

def get_business_context() -> str:
    """Returns full business context from input_schema.json."""
    return json.dumps(INPUT, indent=2)


def get_playwright_boilerplate() -> str:
    """Returns Playwright coding standards and output field spec."""
    return (
        'Standards: async playwright, random delays 1.5-3.5s, '
        'wait_for_load_state networkidle, try/except per section, '
        'screenshot at end. Save to data/raw/{name}_raw.json. '
        'Output schema fields: competitor_name, website, scraped_at, '
        'scrape_status, pricing, features[], homepage_headline, '
        'homepage_usp[], reviews_summary{rating,count,top_pros,top_cons}, '
        'blog_topics[], job_postings{total,top_roles[]}, errors[]'
    )


def save_generated_script(competitor_name: str, script_code: str) -> str:
    """Saves a Playwright script file for a competitor."""
    safe = competitor_name.lower().replace(' ', '_').replace('/', '_')
    p = SCRIPTS_DIR / f'scrape_{safe}.py'
    p.write_text(script_code)
    print(f'  Saved: {p.name} ({p.stat().st_size:,} bytes)')
    return str(p)


def validate_python_syntax(script_code: str) -> str:
    """Validates Python syntax. Returns 'valid' or error message."""
    try:
        ast.parse(script_code)
        return 'valid'
    except SyntaxError as e:
        return f'SyntaxError line {e.lineno}: {e.msg}'


def save_scrape_manifest(manifest: str) -> str:
    """Saves scrape manifest JSON string to disk for Agent 2."""
    p = BASE_DIR / 'scrape_manifest.json'
    p.write_text(manifest)
    print('  Manifest saved')
    return str(p)


# ── Anthropic tool definitions ──────────────────────────────────
TOOLS = [
    {
        "name": "get_business_context",
        "description": "Returns full business context and competitor list from input_schema.json.",
        "input_schema": {"type": "object", "properties": {}, "required": []}
    },
    {
        "name": "get_playwright_boilerplate",
        "description": "Returns Playwright coding standards and output field spec.",
        "input_schema": {"type": "object", "properties": {}, "required": []}
    },
    {
        "name": "save_generated_script",
        "description": "Saves a complete Playwright Python script for one competitor.",
        "input_schema": {
            "type": "object",
            "properties": {
                "competitor_name": {"type": "string", "description": "Competitor name"},
                "script_code": {"type": "string", "description": "Complete Python Playwright script"}
            },
            "required": ["competitor_name", "script_code"]
        }
    },
    {
        "name": "validate_python_syntax",
        "description": "Validates Python syntax of a script. Returns 'valid' or an error message.",
        "input_schema": {
            "type": "object",
            "properties": {
                "script_code": {"type": "string", "description": "Python code to validate"}
            },
            "required": ["script_code"]
        }
    },
    {
        "name": "save_scrape_manifest",
        "description": "Saves the scrape manifest JSON for Agent 2 to use.",
        "input_schema": {
            "type": "object",
            "properties": {
                "manifest": {"type": "string", "description": "JSON string with script list and metadata"}
            },
            "required": ["manifest"]
        }
    },
]

# ── Tool dispatcher ─────────────────────────────────────────────
TOOL_FNS = {
    "get_business_context":   lambda **k: get_business_context(),
    "get_playwright_boilerplate": lambda **k: get_playwright_boilerplate(),
    "save_generated_script":  lambda **k: save_generated_script(**k),
    "validate_python_syntax": lambda **k: validate_python_syntax(**k),
    "save_scrape_manifest":   lambda **k: save_scrape_manifest(**k),
}

print('Agent 1 tools ready:', [t['name'] for t in TOOLS])

## 5. Agentic Loop (Claude)

In [ ]:
def run_claude_agent(system: str, tools: list, tool_fns: dict, prompt: str,
                     model: str = 'claude-opus-4-8', max_tokens: int = 8192) -> str:
    """
    Runs a Claude agentic tool-use loop.
    Keeps looping until stop_reason == 'end_turn', executing tool calls along the way.
    Returns the final text response.
    """
    messages = [{"role": "user", "content": prompt}]
    iteration = 0

    while True:
        iteration += 1
        print(f'  [loop {iteration}] calling Claude...')

        response = client.messages.create(
            model=model,
            max_tokens=max_tokens,
            system=system,
            tools=tools,
            messages=messages
        )

        # Done — return final text
        if response.stop_reason == 'end_turn':
            return next((b.text for b in response.content if hasattr(b, 'text')), '')

        # Tool calls — execute each, feed results back
        if response.stop_reason == 'tool_use':
            messages.append({"role": "assistant", "content": response.content})
            tool_results = []
            for block in response.content:
                if block.type == 'tool_use':
                    print(f'    → tool: {block.name}({list((block.input or {}).keys())})')
                    fn = tool_fns.get(block.name)
                    try:
                        result = fn(**(block.input or {})) if fn else f'Unknown tool: {block.name}'
                    except Exception as e:
                        result = f'Tool error: {e}'
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": str(result)
                    })
            messages.append({"role": "user", "content": tool_results})
        else:
            return f'Unexpected stop_reason: {response.stop_reason}'

## 6. Run Agent 1

In [ ]:
SYSTEM = """
You are Agent 1 — the Prompt Engineer in a competitive intelligence pipeline.

WORKFLOW (follow in order):
1. Call get_business_context() to understand what to scrape
2. Call get_playwright_boilerplate() to get coding standards and output schema
3. For EACH competitor:
   a. Generate a COMPLETE runnable async Playwright Python script
   b. Call validate_python_syntax(script_code) — fix any errors before saving
   c. Call save_generated_script(competitor_name, script_code)
4. Build manifest JSON and call save_scrape_manifest(manifest)

SCRIPT REQUIREMENTS:
- Complete file: imports, async main(), asyncio.run(main()) at bottom
- Scrape: pricing, features, homepage headline + USPs, reviews, blog titles, job counts
- Save output JSON to data/raw/{safe_name}_raw.json
- Save screenshot to data/raw/{safe_name}_screenshot.png
- try/except per section — never crash on missing data
- Random delays between 1.5 and 3.5 seconds
- Realistic user-agent header

MANIFEST FORMAT:
{"generated_at": ISO, "business": name, "scripts": [
  {"competitor": name, "script_path": path, "target_url": url,
   "output_path": path, "priority": "high"|"medium"|"low",
   "scrape_sections": [...]}
]}
"""

PROMPT = (
    'Generate complete async Playwright scraping scripts for ALL competitors '
    'in input_schema.json. Each script must scrape: pricing, features, '
    'homepage messaging, reviews, blog content, job postings. '
    'Validate syntax, save each script, then save the manifest.'
)

print('Running Agent 1 — Prompt Engineer (claude-opus-4-8)')
print('=' * 60)
t0 = datetime.now()

output = run_claude_agent(
    system=SYSTEM,
    tools=TOOLS,
    tool_fns=TOOL_FNS,
    prompt=PROMPT,
    model='claude-opus-4-8',
    max_tokens=8192
)

elapsed = (datetime.now() - t0).seconds
print(f'\nDone in {elapsed}s')
print(output[:600] if output else '(no text output)')

## 7. Verify Output

In [ ]:
print('Generated scripts:')
for f in sorted(SCRIPTS_DIR.glob('*.py')):
    print(f'  {f.name} ({f.stat().st_size:,} bytes)')

manifest_path = BASE_DIR / 'scrape_manifest.json'
if manifest_path.exists():
    m = json.loads(manifest_path.read_text())
    print(f'\nManifest: {len(m.get("scripts", []))} scripts')
    for s in m.get('scripts', []):
        exists = Path(s['script_path']).exists()
        print(f'  {"OK" if exists else "MISSING"}: {s["competitor"]} -> {s["script_path"]}')
else:
    print('\nManifest not found — check agent output above')

In [ ]:
# Preview first generated script
scripts = sorted(SCRIPTS_DIR.glob('*.py'))
if scripts:
    print(scripts[0].name)
    print('=' * 60)
    print(scripts[0].read_text()[:2000])
else:
    print('No scripts generated yet')

## ✅ Done — Next: open `02_agent2_scraper_runner.ipynb`